## Activity Schema Definition and Overlap

**Information Need:** Understand which attributes are associated with each activity type and which of them are shared across activity types.

**Motivation:** Event attributes may differ across activity types, so analysts need to know what information is available for an activity before analyzing it. Attributes also differ in structural scope: some are mandatory, some are populated independently of the activity type, and others only for particular activity types. Knowing an attribute's scope tells analysts whether it is globally applicable or activity-dependent, how to interpret absent values, and whether an analysis involving it should consider the entire log or only selected activity types. Schemas may furthermore overlap, because different activity types can record the same information. Identifying overlaps reveals which information supports comparative or cross-activity analyses, and exposes structural relationships between schemas that inspecting each schema in isolation does not.

**Approach:** Derive the schema of each activity type from the attributes observed in its events, together with their value types and population. Classify each attribute by structural role: `mandatory` or `non-mandatory`, the latter being `global` if consistently filled for all events and otherwise `local`, and local attributes being local either to a single activity type or to several. Compare attribute membership across schemas to identify shared attributes.

**Output:** Per activity type, a schema listing its observed attributes (those taking a non-null value), their value types, and their population across its events. A classification of all attributes as mandatory, global, or local, local ones further marked as pertaining to one or to several activity types. A list of the attributes shared by multiple activity types, with the number of events in which each activity type populates them.

In [ ]:
import pandas as pd
import pm4py

from ipywidgets import interact

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

#rename in case of changes above
MANDATORY_COLUMNS = {CASE_ID, ACTIVITY, TIMESTAMP}

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

### Activity Schema

In [ ]:
activity_stats = event_log[ACTIVITY].value_counts()


@interact(activity=list(activity_stats.index))
def compute_schema(activity):
    df = event_log[event_log[ACTIVITY] == activity]
    df = df.dropna(axis=1, how='all') # drop all completely empty columns
    df = df.drop(columns=[ACTIVITY, CASE_ID, TIMESTAMP])  # drop standard columns
    return df.info()

### Attribute Scope

In [ ]:
mandatory_columns = [c for c in event_log.columns if c in MANDATORY_COLUMNS]

remaining_columns = [c for c in event_log.columns if c not in MANDATORY_COLUMNS]
fully_populated = event_log[remaining_columns].notna().all()

global_columns = fully_populated[fully_populated].index.tolist()
local_columns = fully_populated[~fully_populated].index.tolist()

print(f'Mandatory columns ({len(mandatory_columns)}):')
print(mandatory_columns)
print(f'\nGlobal columns ({len(global_columns)}):')
print(global_columns)
print(f'\nLocal columns ({len(local_columns)}):')
print(local_columns)

In [ ]:
activities_per_column = {
    column: event_log.loc[event_log[column].notna(), ACTIVITY].nunique()
    for column in local_columns
}

multiple_activities = [column for column, n_activities in activities_per_column.items() if n_activities > 1]
single_activities = [column for column, n_activities in activities_per_column.items() if n_activities == 1]

print(f'Local to multiple activities ({len(multiple_activities)}):')
print(multiple_activities)
print(f'\nLocal to a single activity ({len(single_activities)}):')
print(single_activities)

In [ ]:
#Display local attributes to a specific activity type

activity_to_single_activities = {activity: [] for activity in event_log[ACTIVITY].unique()}

for column in single_activities:
    activity = event_log.loc[event_log[column].notna(), ACTIVITY].iloc[0]
    activity_to_single_activities[activity].append(column)

@interact(activity=sorted(activity_to_single_activities.keys()))
def show_single_activities(activity):
    return activity_to_single_activities[activity]

### Schema Overlap

In [ ]:
data = []

#compute matrix of number of attribute values for each activity
for activity in list(activity_stats.index):
    filtered_log = event_log[event_log[ACTIVITY] == activity]
    counts = { col: filtered_log[col].count() for col in event_log.columns}
    data.append(counts)
df = pd.DataFrame.from_records(data, index=activity_stats.index)

# Compute simplified table to identify overlaps in populated attribute values: drop all columns where only one row is not 0 and where no row is zero
# Activities with all 0s don't show any overlaps with the schemas of other activities
no_rows = len(df)
cols_partial = [col for col in df.columns if 2 <= (df[col] != 0).sum() < no_rows]

display(df[cols_partial])